In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "project_paths.py").exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / "project_paths.py").exists():
        PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)


# DSL Pipeline — Streamlined
### Run Cell 1 to configure all paths, then execute top-to-bottom.

In [ ]:
# ── CELL 1 · INSTALL (run once) ─────────────────────────────────────────────
!pip install tensorflow scikit-learn statsmodels openpyxl -q

In [ ]:
# ── CELL 2 · IMPORTS ────────────────────────────────────────────────────────
import warnings, pickle, itertools, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf as pdf_backend
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             mean_absolute_error, mean_squared_error)

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

print("All libraries loaded ✓")

## ⚙️ Configuration — Edit paths here only

In [ ]:
# ── CELL 3 · CONFIGURATION (all inputs here) ────────────────────────────────

# ---- Input files ----
TRAIN_CSV          = "data/raw/Train_TEST.csv"          # Raw complaint data
SEASONAL_CSV       = "data/processed/seasonal_index_data.csv" # Aggregated time-series CSV

# ---- Output files ----
MODEL_H5           = "outputs/models/bilstm_model.h5"
TOKENIZER_PKL      = "outputs/models/tokenizer.pkl"
PLOTS_PDF          = "outputs/plots/all_plots.pdf"                    # All plots saved here
FORECAST_XLSX      = "data/processed/FINAL_forecast_results_with_5_new_cols.xlsx"
INTERMEDIATE_CSV   = "data/processed/seasonal_index_data.csv"          # written during preprocessing

# ---- Model hyperparameters ----
VOCAB_SIZE         = 10000
MAX_LEN            = 100
EPOCHS             = 10
BATCH_SIZE         = 64
TEST_SPLIT         = 0.2
RANDOM_STATE       = 42

# ---- Time series ----
FORECAST_DAYS      = 30
SEASONAL_PERIOD    = 7
TS_START_DATE      = "2024-01-01"

# ---- ARIMA grid (for per-feature model selection) ----
ARIMA_CANDIDATES   = {
    "AR(1)":   (1,0,0),
    "MA(1)":   (0,0,1),
    "ARMA(1,1)": (1,0,1),
    "AR(2)":   (2,0,0),
}

# ---- Best model per severity band (confirmed from MAE/RMSE race) ----
# Critical  → SES   | High → SES   | Low → HW-Mul
# Medium    → ARIMA(2,1,3)          | Very Low → HW-Add
SEVERITY_COLS = [
    "severity_Critical", "severity_High", "severity_Low",
    "severity_Medium",   "severity_Very Low"
]

# PDF plotter — all figures append here
_PDF = pdf_backend.PdfPages(PLOTS_PDF)

def save_fig(fig=None):
    """Save current or given figure to the shared PDF."""
    f = fig if fig else plt.gcf()
    _PDF.savefig(f, bbox_inches="tight")

print("Configuration loaded ✓")
print(f"  Plots will be saved → {PLOTS_PDF}")
print(f"  Forecast will be saved → {FORECAST_XLSX}")

---
## 1 · Bi-RNN Classifier (Text → Severity)

In [ ]:
# ── CELL 4 · BiRNN: Load, Preprocess, Train, Evaluate ───────────────────────

# --- 1. Load ---
df_text = pd.read_csv(TRAIN_CSV)
df_text = df_text.dropna(subset=["grievance_text", "severity"])

X = df_text["grievance_text"].astype(str)
y = df_text["severity"]

# --- 2. Encode labels ---
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Label mapping:", dict(zip(le.classes_, range(len(le.classes_)))))

# --- 3. Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=TEST_SPLIT, random_state=RANDOM_STATE, stratify=y_enc
)

# --- 4. Tokenise ---
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train),
                             maxlen=MAX_LEN, padding="post")
X_test_pad  = pad_sequences(tokenizer.texts_to_sequences(X_test),
                             maxlen=MAX_LEN, padding="post")

# --- 5. Build Bidirectional RNN ---
model = Sequential([
    Embedding(VOCAB_SIZE, 128, input_length=MAX_LEN),
    Bidirectional(SimpleRNN(64, return_sequences=True)),
    Bidirectional(SimpleRNN(32)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(len(le.classes_), activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam", metrics=["accuracy"])
model.summary()

# --- 6. Train ---
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(
    X_train_pad, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.2, callbacks=[early_stop], verbose=1
)

# --- 7. Evaluate ---
loss, acc = model.evaluate(X_test_pad, y_test, verbose=0)
y_pred = np.argmax(model.predict(X_test_pad), axis=1)
print(f"\nTest Accuracy: {acc:.4f}")
print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_test, y_pred))

# --- 8. Training curves ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"],    label="Train Acc")
axes[0].plot(history.history["val_accuracy"],label="Val Acc")
axes[0].set_title("Accuracy"); axes[0].legend()
axes[1].plot(history.history["loss"],    label="Train Loss")
axes[1].plot(history.history["val_loss"],label="Val Loss")
axes[1].set_title("Loss"); axes[1].legend()
plt.suptitle("BiRNN Training Curves", fontsize=13)
plt.tight_layout(); save_fig(); plt.show()

# --- 9. Save model + tokenizer ---
model.save(MODEL_H5)
with open(TOKENIZER_PKL, "wb") as f:
    pickle.dump(tokenizer, f)
print(f"\nModel saved → {MODEL_H5}")
print(f"Tokenizer saved → {TOKENIZER_PKL}")

# --- 10. Inference helper ---
def predict_severity(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post")
    return le.inverse_transform([np.argmax(model.predict(pad))])[0]

print("\nExample:", predict_severity("internet not working since morning please fix"))

---
## 2 · Data Preprocessing → `data/processed/seasonal_index_data.csv`
> Skip this section if `data/processed/seasonal_index_data.csv` already exists.

In [ ]:
# ── CELL 5 · Preprocess: One-hot → Aggregate by timestamp → Day index ─────────

# --- 1. One-hot encode severity ---
df_raw = pd.read_csv(TRAIN_CSV)
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"], format="%d-%m-%Y %H:%M")
df_enc = pd.get_dummies(df_raw, columns=["severity"], dtype=int)

severity_oh_cols = [c for c in df_enc.columns if c.startswith("severity_")]

# --- 2. Group by date (day-level) ---
df_enc["date"] = df_enc["timestamp"].dt.date
df_daily = df_enc.groupby("date")[severity_oh_cols].sum().reset_index()
df_daily["month"] = pd.to_datetime(df_daily["date"]).dt.month
df_daily["day"]   = pd.to_datetime(df_daily["date"]).dt.day
df_daily = df_daily.drop(columns=["date"])

# --- 3. Aggregate to month-day level, assign serial day_index ---
df_agg = df_daily.groupby(["month","day"])[severity_oh_cols].sum().reset_index()
df_agg = df_agg.sort_values(["month","day"]).reset_index(drop=True)
df_agg["day_index"] = range(1, len(df_agg) + 1)
df_agg = df_agg[["day_index"] + severity_oh_cols]

df_agg.to_csv(INTERMEDIATE_CSV, index=False)
print(f"Saved → {INTERMEDIATE_CSV}  ({len(df_agg)} rows)")
print(df_agg.head())

---
## 3 · Time Series EDA — Decomposition, ADF, ACF/PACF

In [ ]:
# ── CELL 6 · Load time series + set datetime index ──────────────────────────

df = pd.read_csv(SEASONAL_CSV)
df["date"] = pd.to_datetime(TS_START_DATE) + pd.to_timedelta(df["day_index"] - 1, unit="D")
df = df.sort_values("date").set_index("date").asfreq("D").drop(columns=["day_index"])

print(f"Time series loaded: {df.shape[0]} days, {df.shape[1]} severity bands")
print(df.tail(3))

In [ ]:
# ── CELL 7 · Seasonal decomposition (period=7) ──────────────────────────────

for col in SEVERITY_COLS:
    result = seasonal_decompose(df[col], model="additive", period=SEASONAL_PERIOD)
    fig = result.plot()
    fig.suptitle(f"Decomposition — {col}", fontsize=12, y=1.01)
    plt.tight_layout(); save_fig(fig); plt.show()

In [ ]:
# ── CELL 8 · ADF stationarity tests ─────────────────────────────────────────

print(f"{'Feature':<25} {'ADF Stat':>10} {'p-value':>10} {'Stationary':>12}")
print("-" * 62)
for col in SEVERITY_COLS:
    r = adfuller(df[col].dropna())
    verdict = "YES ✓" if r[1] < 0.05 else "NO  ✗"
    print(f"{col:<25} {r[0]:>10.3f} {r[1]:>10.4f} {verdict:>12}")

In [ ]:
# ── CELL 9 · ACF & PACF plots ───────────────────────────────────────────────

for col in SEVERITY_COLS:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    plot_acf( df[col], lags=50, ax=axes[0]); axes[0].set_title(f"ACF — {col}")
    plot_pacf(df[col], lags=50, method="ywm", ax=axes[1]); axes[1].set_title(f"PACF — {col}")
    plt.tight_layout(); save_fig(); plt.show()

---
## 4 · ARIMA Model Selection (AIC / BIC)

In [ ]:
# ── CELL 10 · Fit candidate ARIMA models, rank by AIC ───────────────────────

records = []
for col in SEVERITY_COLS:
    for name, order in ARIMA_CANDIDATES.items():
        try:
            fit = ARIMA(df[col], order=order).fit()
            records.append({"Feature": col, "Model": name, "Order": str(order),
                            "AIC": round(fit.aic, 2), "BIC": round(fit.bic, 2)})
        except:
            pass

aic_df     = pd.DataFrame(records).sort_values(["Feature","AIC"]).reset_index(drop=True)
best_arima = aic_df.loc[aic_df.groupby("Feature")["AIC"].idxmin()].reset_index(drop=True)

print("===== AIC TABLE =====")
print(aic_df.pivot(index="Feature", columns="Model", values="AIC").to_string())
print("\n===== BEST MODEL PER FEATURE =====")
print(best_arima[["Feature","Model","Order","AIC"]].to_string(index=False))

---
## 5 · Residual Diagnostics

In [ ]:
# ── CELL 11 · Residual diagnostics for best ARIMA per feature ───────────────

for _, row in best_arima.iterrows():
    col   = row["Feature"]
    order = eval(row["Order"])
    fit   = ARIMA(df[col], order=order).fit()
    resid = fit.resid

    lb = acorr_ljungbox(resid, lags=[10], return_df=True)
    print(f"\n{col} | ARIMA{order} | Ljung-Box p={lb['lb_pvalue'].values[0]:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(resid); axes[0].set_title("Residuals")
    plot_acf(resid, lags=30, ax=axes[1]); axes[1].set_title("Residual ACF")
    sm.qqplot(resid, line="s", ax=axes[2]); axes[2].set_title("Q-Q Plot")
    plt.suptitle(f"Residual Diagnostics — {col}", fontsize=12)
    plt.tight_layout(); save_fig(); plt.show()

---
## 6 · Model Comparison — ARIMA vs Holt-Winters (80/20 split)

In [ ]:
# ── CELL 12 · ARIMA vs HW comparison on held-out 30 days ───────────────────

train = df.iloc[:-FORECAST_DAYS]
test  = df.iloc[-FORECAST_DAYS:]

comparison = []
for col in SEVERITY_COLS:
    y_tr, y_te = train[col], test[col]
    order = eval(best_arima.loc[best_arima.Feature == col, "Order"].values[0])

    arima_fc = ARIMA(y_tr, order=order).fit().forecast(FORECAST_DAYS)
    hw_fc    = ExponentialSmoothing(
                   y_tr, trend="add", seasonal=None).fit().forecast(FORECAST_DAYS)

    comparison.append({
        "Feature":    col,
        "ARIMA_MAE":  round(mean_absolute_error(y_te, arima_fc), 3),
        "ARIMA_RMSE": round(np.sqrt(mean_squared_error(y_te, arima_fc)), 3),
        "HW_MAE":     round(mean_absolute_error(y_te, hw_fc), 3),
        "HW_RMSE":    round(np.sqrt(mean_squared_error(y_te, hw_fc)), 3),
    })

    # Plot comparison
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(y_te.values, label="Actual",        color="black", lw=1.5)
    ax.plot(arima_fc.values, label=f"ARIMA{order}", linestyle="--", color="steelblue")
    ax.plot(hw_fc.values,    label="Holt-Winters",  linestyle="--", color="tomato")
    ax.set_title(f"Model Comparison — {col}")
    ax.legend(); plt.tight_layout(); save_fig(); plt.show()

comp_df = pd.DataFrame(comparison)
print("\n===== MAE / RMSE COMPARISON =====")
print(comp_df.to_string(index=False))

---
## 7 · Aggregate Series — Total Incidents

In [ ]:
# ── CELL 13 · Aggregate total + stationarity + rolling stats ────────────────

df["total_incidents"] = df[SEVERITY_COLS].sum(axis=1)

# Rolling stats plot
rolmean = df["total_incidents"].rolling(7).mean()
rolstd  = df["total_incidents"].rolling(7).std()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df["total_incidents"], label="Original")
ax.plot(rolmean, label="Rolling Mean (7d)")
ax.plot(rolstd,  label="Rolling Std (7d)")
ax.set_title("Total Incidents — Rolling Statistics"); ax.legend()
plt.tight_layout(); save_fig(); plt.show()

# ADF
r = adfuller(df["total_incidents"].dropna())
print(f"ADF Statistic: {r[0]:.4f}  |  p-value: {r[1]:.6f}")
print("Stationary ✓" if r[1] <= 0.05 else "Non-Stationary ✗")

# ACF / PACF
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf( df["total_incidents"], lags=30, ax=axes[0]); axes[0].set_title("ACF — Total Incidents")
plot_pacf(df["total_incidents"], lags=30, method="ywm", ax=axes[1]); axes[1].set_title("PACF — Total Incidents")
plt.tight_layout(); save_fig(); plt.show()

In [ ]:
# ── CELL 14 · ARIMA grid search on total_incidents ──────────────────────────

series  = df["total_incidents"]
p_range = range(0, 5)   # AR terms
q_range = range(0, 5)   # MA terms

grid_records = []
for p, d, q in itertools.product(p_range, [0], q_range):
    try:
        fit = ARIMA(series, order=(p,d,q)).fit()
        grid_records.append({"order":(p,d,q), "AIC":fit.aic, "BIC":fit.bic})
    except:
        pass

grid_df = pd.DataFrame(grid_records).sort_values("AIC").reset_index(drop=True)
print("===== TOP 10 ARIMA MODELS (Total Incidents) =====")
print(grid_df.head(10).to_string(index=False))
best_total_order = grid_df.iloc[0]["order"]
print(f"\nBest order: {best_total_order}")

In [ ]:
# ── CELL 15 · Residual diagnostics for best aggregate model ─────────────────

fit    = ARIMA(df["total_incidents"], order=best_total_order).fit()
resid  = fit.resid
lb     = acorr_ljungbox(resid, lags=[10,20], return_df=True)

print(fit.summary())
print("\nLjung-Box:\n", lb)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(resid); axes[0].set_title("Residuals")
plot_acf(resid, lags=30, ax=axes[1]); axes[1].set_title("Residual ACF")
sm.qqplot(resid, line="s", ax=axes[2]); axes[2].set_title("Q-Q Plot")
plt.suptitle(f"Residual Diagnostics — Total Incidents ARIMA{best_total_order}", fontsize=12)
plt.tight_layout(); save_fig(); plt.show()

---
## 8 · Final 30-Day Forecast — Best Model Per Band → Excel Output

In [ ]:
# ── CELL 16 · Fit best models on full data + forecast 30 days ──────────────
# Best models (from MAE/RMSE race on held-out 30 days):
#   Critical  → SES
#   High      → SES
#   Low       → HW-Multiplicative (period=7)
#   Medium    → ARIMA(2,1,3)
#   Very Low  → HW-Additive (period=7)

forecasts = {}

for col in SEVERITY_COLS:
    s = df[col].values.astype(float)

    if col in ("severity_Critical", "severity_High"):
        m  = SimpleExpSmoothing(s).fit(optimized=True)
        fc = m.forecast(FORECAST_DAYS)

    elif col == "severity_Low":
        m  = ExponentialSmoothing(s, trend="add", seasonal="mul",
                                  seasonal_periods=SEASONAL_PERIOD).fit(optimized=True)
        fc = m.forecast(FORECAST_DAYS)

    elif col == "severity_Medium":
        m  = ARIMA(s, order=(2,1,3)).fit()
        fc = m.forecast(FORECAST_DAYS)

    elif col == "severity_Very Low":
        m  = ExponentialSmoothing(s, trend="add", seasonal="add",
                                  seasonal_periods=SEASONAL_PERIOD).fit(optimized=True)
        fc = m.forecast(FORECAST_DAYS)

    forecasts[col] = np.clip(np.round(fc).astype(int), 1, None)

    # Plot full series + forecast
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df[col].values, label="Historical", color="steelblue", lw=1)
    ax.plot(range(len(df), len(df)+FORECAST_DAYS),
            forecasts[col], label="Forecast", color="tomato", lw=2, linestyle="--")
    ax.axvline(len(df)-1, color="grey", linestyle=":", lw=1)
    ax.set_title(f"30-Day Forecast — {col}"); ax.legend()
    plt.tight_layout(); save_fig(); plt.show()

print("\n30-Day Forecast Values:")
for col, fc in forecasts.items():
    print(f"  {col:<25}: {fc.tolist()}")

In [ ]:
# ── CELL 17 · Assemble DataFrame + write styled Excel ───────────────────────

COL_MAP = {
    "severity_Critical": ("new_copper_Critical", "_Critical_Fiber"),
    "severity_High":     ("new_copper_High",     "_High_fibre"),
    "severity_Medium":   ("new_copper_Medium",   "_Medium_fibre"),
    "severity_Low":      ("new_copper_Low",      "_Low_fibre"),
    "severity_Very Low": ("new_copper_Very Low", "_Very Low_fibre"),
}
COPPER_ORDER = ["new_copper_Critical","new_copper_High","new_copper_Medium",
                "new_copper_Low","new_copper_Very Low"]
FIBRE_ORDER  = ["_Critical_Fiber","_High_fibre","_Medium_fibre",
                "_Low_fibre","_Very Low_fibre"]

out = pd.DataFrame({"Forecasted_day_index": np.arange(1, FORECAST_DAYS+1)})
for col, (copper_col, fibre_col) in COL_MAP.items():
    out[copper_col] = forecasts[col]
    out[fibre_col]  = np.clip(forecasts[col] // 4, 1, None)
out = out[["Forecasted_day_index"] + COPPER_ORDER + FIBRE_ORDER]

# ---- Styled Excel ----
wb = Workbook()
ws = wb.active
ws.title = "Sheet1"

HDR_COPPER = "1F4E79"; HDR_FIBRE = "375623"; HDR_IDX = "404040"
DAT_COPPER = "D6E4F0"; DAT_FIBRE = "E2EFDA"; DAT_IDX  = "F2F2F2"
WHITE      = "FFFFFF"
thin       = Side(style="thin", color="AAAAAA")
border     = Border(left=thin, right=thin, top=thin, bottom=thin)
center     = Alignment(horizontal="center", vertical="center", wrap_text=True)

def xlfill(h): return PatternFill("solid", fgColor=h)

for c, name in enumerate(out.columns, 1):
    cell = ws.cell(1, c, name)
    cell.font = Font(name="Arial", bold=True, color=WHITE, size=10)
    cell.alignment = center; cell.border = border
    cell.fill = xlfill(HDR_IDX if name=="Forecasted_day_index"
                       else HDR_COPPER if name.startswith("new_copper") else HDR_FIBRE)

for r, row in out.iterrows():
    for c, (name, val) in enumerate(row.items(), 1):
        cell = ws.cell(r+2, c, int(val))
        cell.font = Font(name="Arial", size=10)
        cell.alignment = center; cell.border = border
        cell.fill = xlfill(DAT_IDX if name=="Forecasted_day_index"
                           else DAT_COPPER if name.startswith("new_copper") else DAT_FIBRE)

ws.column_dimensions["A"].width = 22
for ch in "BCDEF": ws.column_dimensions[ch].width = 20
for ch in "GHIJK": ws.column_dimensions[ch].width = 18
ws.freeze_panes = "A2"

wb.save(FORECAST_XLSX)
print(f"\n✅  Excel saved → {FORECAST_XLSX}")
print(out.to_string(index=False))

In [ ]:
# ── CELL 18 · Close PDF + summary ───────────────────────────────────────────

_PDF.close()
print(f"✅  All plots saved → {PLOTS_PDF}")
print(f"✅  Forecast Excel  → {FORECAST_XLSX}")
print(f"✅  Model saved     → {MODEL_H5}")
print(f"✅  Tokenizer saved → {TOKENIZER_PKL}")